# Day 3 — Performance Analytics
Daily returns, CAGR, Sharpe, Sortino, Alpha/Beta, Max Drawdown, Fund Scorecard, Benchmark Comparison

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

BASE = os.path.dirname(os.path.abspath('.'))
PROC = 'data/processed'
RF   = 0.065 / 252

nav = pd.read_csv(f'{PROC}/nav_history_clean.csv', parse_dates=['date'])
fm  = pd.read_csv(f'{PROC}/fund_master_clean.csv')
er  = pd.read_csv(f'{PROC}/expense_ratio_clean.csv')
bm  = pd.read_csv(f'{PROC}/benchmark_data_clean.csv', parse_dates=['date'])

nav = nav.sort_values(['scheme_code','date']).reset_index(drop=True)
scheme_names = fm.set_index('scheme_code')['scheme_name'].to_dict()
nav['name'] = nav['scheme_code'].map(scheme_names)
print(f'nav shape: {nav.shape}')

## 1. Daily Returns

In [ ]:
nav['daily_ret'] = nav.groupby('scheme_code')['nav'].pct_change()
nav = nav.dropna(subset=['daily_ret'])

# distribution check
sample = nav[nav['scheme_code'] == nav['scheme_code'].iloc[0]]['daily_ret']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sample, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Daily Return Distribution (sample fund)')
axes[0].set_xlabel('Daily Return')
stats.probplot(sample, plot=axes[1])
axes[1].set_title('Q-Q Plot')
plt.tight_layout()
plt.show()

print('Return stats:')
print(nav.groupby('scheme_code')['daily_ret'].agg(['mean','std','skew']).round(6).head(5))

## 2. CAGR — 1yr, 3yr, 5yr

In [ ]:
def cagr(series, years):
    end   = series.iloc[-1]
    start = series.iloc[max(0, len(series) - int(years * 252))]
    return (end / start) ** (1 / years) - 1 if start > 0 else np.nan

rows = []
for code, grp in nav.groupby('scheme_code'):
    grp = grp.sort_values('date')
    rows.append({'scheme_code': code, 'scheme_name': scheme_names.get(code, str(code)),
                 'cagr_1y': round(cagr(grp['nav'], 1)*100, 2),
                 'cagr_3y': round(cagr(grp['nav'], 3)*100, 2),
                 'cagr_5y': round(cagr(grp['nav'], 5)*100, 2)})
cagr_df = pd.DataFrame(rows).sort_values('cagr_3y', ascending=False)
print(cagr_df[['scheme_name','cagr_1y','cagr_3y','cagr_5y']].to_string(index=False))

## 3. Sharpe Ratio

In [ ]:
def sharpe(rets):
    excess = rets - RF
    return (excess.mean() / rets.std()) * np.sqrt(252) if rets.std() > 0 else np.nan

sharpe_df = nav.groupby('scheme_code')['daily_ret'].apply(sharpe).rename('sharpe').reset_index()
sharpe_df['scheme_name'] = sharpe_df['scheme_code'].map(scheme_names)
sharpe_df = sharpe_df.sort_values('sharpe', ascending=False).reset_index(drop=True)
sharpe_df['sharpe_rank'] = sharpe_df.index + 1
print(sharpe_df[['scheme_name','sharpe','sharpe_rank']].to_string(index=False))

## 4. Sortino Ratio

In [ ]:
def sortino(rets):
    excess   = rets - RF
    downside = rets[rets < 0].std()
    return (excess.mean() / downside) * np.sqrt(252) if downside > 0 else np.nan

sortino_df = nav.groupby('scheme_code')['daily_ret'].apply(sortino).rename('sortino').reset_index()
sortino_df['scheme_name'] = sortino_df['scheme_code'].map(scheme_names)
sortino_df = sortino_df.sort_values('sortino', ascending=False).reset_index(drop=True)
sortino_df['sortino_rank'] = sortino_df.index + 1
print(sortino_df[['scheme_name','sortino','sortino_rank']].to_string(index=False))

## 5. Alpha & Beta (OLS vs Nifty 100)

In [ ]:
bm_daily = bm.set_index('date')['nifty100'].pct_change().dropna().rename('bm_ret')

ab_rows = []
for code, grp in nav.groupby('scheme_code'):
    grp = grp.set_index('date')['daily_ret'].dropna()
    merged = pd.concat([grp, bm_daily], axis=1).dropna()
    if len(merged) < 30: continue
    slope, intercept, r, p, se = stats.linregress(merged['bm_ret'], merged['daily_ret'])
    ab_rows.append({'scheme_code': code, 'scheme_name': scheme_names.get(code, str(code)),
                    'beta': round(slope, 4), 'alpha_annual': round(intercept*252*100, 4),
                    'r_squared': round(r**2, 4)})
alpha_beta = pd.DataFrame(ab_rows).sort_values('alpha_annual', ascending=False)
print(alpha_beta[['scheme_name','alpha_annual','beta','r_squared']].to_string(index=False))
alpha_beta.to_csv('alpha_beta.csv', index=False)

## 6. Maximum Drawdown

In [ ]:
dd_rows = []
for code, grp in nav.groupby('scheme_code'):
    grp = grp.sort_values('date')
    roll_max = grp['nav'].cummax()
    dd = grp['nav'] / roll_max - 1
    min_idx  = dd.idxmin()
    peak_idx = grp.loc[:min_idx, 'nav'].idxmax()
    dd_rows.append({'scheme_code': code, 'scheme_name': scheme_names.get(code, str(code)),
                    'max_dd': round(dd.min()*100, 2),
                    'peak_date': grp.loc[peak_idx, 'date'].date(),
                    'trough_date': grp.loc[min_idx, 'date'].date()})
dd_df = pd.DataFrame(dd_rows).sort_values('max_dd')
print(dd_df[['scheme_name','max_dd','peak_date','trough_date']].to_string(index=False))

## 7. Fund Scorecard (0–100)

In [ ]:
sc = cagr_df[['scheme_code','cagr_3y']].merge(sharpe_df[['scheme_code','sharpe','sharpe_rank']], on='scheme_code')
sc = sc.merge(sortino_df[['scheme_code','sortino_rank']], on='scheme_code')
sc = sc.merge(alpha_beta[['scheme_code','alpha_annual']], on='scheme_code')
sc = sc.merge(dd_df[['scheme_code','max_dd']], on='scheme_code')
sc = sc.merge(er[['scheme_code','direct_ter']], on='scheme_code', how='left')

n = len(sc)
sc['ret_rank']   = sc['cagr_3y'].rank(ascending=True)
sc['alpha_rank'] = sc['alpha_annual'].rank(ascending=True)
sc['er_rank']    = sc['direct_ter'].rank(ascending=False)
sc['dd_rank']    = sc['max_dd'].rank(ascending=False)
sc['score'] = (0.30*sc['ret_rank'] + 0.25*sc['sharpe_rank'] + 0.20*sc['alpha_rank'] +
               0.15*sc['er_rank']  + 0.10*sc['dd_rank']) / n * 100
sc['score'] = sc['score'].round(2)
sc['scheme_name'] = sc['scheme_code'].map(scheme_names)
scorecard = sc[['scheme_code','scheme_name','score','cagr_3y','sharpe','alpha_annual','direct_ter','max_dd']]
scorecard = scorecard.sort_values('score', ascending=False).reset_index(drop=True)
scorecard.index += 1
print(scorecard.to_string())
scorecard.to_csv('fund_scorecard.csv')

## 8. Benchmark Comparison Chart

In [ ]:
top5   = scorecard.head(5)['scheme_code'].tolist()
cutoff = nav['date'].max() - pd.DateOffset(years=3)
colors = ['#4fc3f7','#66bb6a','#ffa726','#ef5350','#ce93d8']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for i, code in enumerate(top5):
    grp = nav[(nav['scheme_code']==code) & (nav['date']>=cutoff)].set_index('date')['nav']
    grp = grp / grp.iloc[0] * 100
    ax.plot(grp.index, grp.values, color=colors[i], lw=1.5,
            label=scheme_names.get(code,'').split(' Fund')[0][:22])
for col, lbl, ls in [('nifty50','Nifty 50','--'),('nifty100','Nifty 100',':')]:
    b = bm[bm['date']>=cutoff].set_index('date')[col].dropna()
    ax.plot(b.index, b/b.iloc[0]*100, color='grey', lw=1.2, ls=ls, label=lbl)
ax.set_title('Top 5 Funds vs Benchmark (3yr, rebased 100)')
ax.legend(fontsize=7)

ax2 = axes[1]
te_vals, te_labels = [], []
for code in top5:
    grp = nav[(nav['scheme_code']==code) & (nav['date']>=cutoff)].set_index('date')['daily_ret']
    b   = bm[bm['date']>=cutoff].set_index('date')['nifty100'].pct_change().dropna()
    m   = pd.concat([grp, b.rename('bm')], axis=1).dropna()
    te  = (m['daily_ret'] - m['bm']).std() * np.sqrt(252) * 100
    te_vals.append(round(te, 2))
    te_labels.append(scheme_names.get(code,'').split(' Fund')[0][:20])
ax2.barh(te_labels, te_vals, color=colors[:len(te_vals)])
ax2.set_title('Tracking Error vs Nifty 100 (annualised %)')
ax2.set_xlabel('Tracking Error (%)')

plt.tight_layout()
plt.savefig('dashboard/benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('chart saved')